## Celda 1 — import

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
import json

## Celda 2 — Configuración (rutas + columnas), Cargar CSV, filtrar etiquetados, mapear labels

In [ ]:
DATA_DIR = Path(r"C:\Users\leona\Documents\Thesis_Project_UACH\Temp\Dataset\BeesAnna\sound_files")
CSV_PATH = Path(r"C:\Users\leona\Documents\Thesis_Project_UACH\Temp\Dataset\all_data_updated.csv")

ID_COL = "file name"            # <-- columna de nombres
TARGET_COL = "queen status"     # <-- la etiqueta que se utilizará
VALID_CLASSES = {0, 1, 2, 3}

# 1) Construir diccionario: nombre_archivo.wav -> ruta_completa
wav_paths = sorted(list(DATA_DIR.rglob("*.wav"))) + sorted(list(DATA_DIR.rglob("*.WAV")))
name_to_path = {p.name: str(p) for p in wav_paths}  # str() para que sea serializable y fácil

df = pd.read_csv(CSV_PATH)

assert ID_COL in df.columns, f"Falta ID_COL={ID_COL} en el CSV"
assert TARGET_COL in df.columns, f"Falta TARGET_COL={TARGET_COL} en el CSV"

# 2) Normaliza nombres (solo basename)
df["_basename"] = df[ID_COL].astype(str).map(lambda s: Path(s).name)

# 3) Convertir queen status a numérico y filtrar clases válidas
df["_y"] = pd.to_numeric(df[TARGET_COL], errors="coerce")
labeled = df[df["_y"].isin([0,1,2,3])].copy()
labeled["y"] = labeled["_y"].astype(np.int64)

# 4) Crear columna "path" (ruta completa al wav)
labeled["path"] = labeled["_basename"].map(name_to_path)

# 5) Verificar que no falte ninguno
missing = labeled["path"].isna().sum()
print("Etiquetados:", len(labeled), "| faltantes sin path:", int(missing))
assert missing == 0, "Hay archivos etiquetados que no se encontraron en la carpeta."
print("Conteo por clase:\n", labeled["y"].value_counts().sort_index())

Etiquetados: 1275 | faltantes sin path: 0
Conteo por clase:
 y
0    179
1    158
2    259
3    679
Name: count, dtype: int64


In [ ]:
# Audio / MFCC params (igual que la prueba "TestMFCC")
SR = 16000
TRIM_DB = 30
SEG_SEC = 2.0
HOP_SEC = 1.0
N_MFCC = 32
N_FFT  = int(0.025 * SR)
HOP_LEN= int(0.010 * SR)
FMIN, FMAX = 20, SR//2
ADD_DELTAS = True
RANDOM_SEED = 123
TEST_SIZE = 0.15
VAL_SIZE  = 0.15

In [4]:
OUT_DIR = Path(r"C:\Users\leona\Documents\Thesis_Project_UACH\Temp\Dataset\features_mfcc_labeled_03-3")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Celda 3 — Funciones: limpiar, segmentar, MFCC (+Δ +ΔΔ)

In [5]:
def peak_normalize(x, eps=1e-9):
    return x / (np.max(np.abs(x)) + eps)

def load_and_clean(path):
    """
    Preprocesamiento a nivel de archivo (CONTRATO DE PREPROCESAMIENTO):
    1. Load + resample a SR=16000
    2. Trim con TRIM_DB=30 (elimina silencios al inicio/fin)
    3. Peak normalization sobre el audio TRIMMEADO (a nivel de archivo completo)
    """
    x, _ = librosa.load(str(path), sr=SR, mono=True)
    x, _ = librosa.effects.trim(x, top_db=TRIM_DB)
    x = peak_normalize(x)
    return x

def segment_signal(x, sr, seg_sec, hop_sec):
    seg_len = int(seg_sec * sr)
    hop_len = int(hop_sec * sr)
    if len(x) < seg_len:
        x = np.pad(x, (0, seg_len - len(x)), mode="reflect")
    segments = []
    for start in range(0, max(1, len(x)-seg_len+1), hop_len):
        seg = x[start:start+seg_len]
        if len(seg) < seg_len:
            seg = np.pad(seg, (0, seg_len - len(seg)), mode="reflect")
        segments.append(seg)
    return segments

def mfcc_features(seg):
    """
    Extrae MFCCs + deltas con C0 eliminado.
    Contrato de preprocesamiento:
    1. Se recibe 'seg' ya trimmeado, normalizado y segmentado
    2. Extraemos 33 MFCCs (n_mfcc=33)
    3. Eliminamos C0 → queda (32, T)
    4. Calculamos Δ y ΔΔ sobre los 32 coefs restantes
    5. Apilamos → (3, 32, T)
    """
    # Extraer 33 MFCCs para luego eliminar C0
    mfcc = librosa.feature.mfcc(
        y=seg, sr=SR, n_mfcc=33, n_fft=N_FFT, hop_length=HOP_LEN,
        fmin=FMIN, fmax=FMAX
    )
    # Eliminar C0 (primer coeficiente) → ahora (32, T)
    mfcc = mfcc[1:, :]
    
    if ADD_DELTAS:
        d1 = librosa.feature.delta(mfcc)
        d2 = librosa.feature.delta(mfcc, order=2)
        feat = np.stack([mfcc, d1, d2], axis=0)  # (3, 32, T)
    else:
        feat = mfcc[np.newaxis, :, :]            # (1, 32, T)
    return feat.astype(np.float32)

## Celda 4 — Split estratificado

In [6]:
paths = labeled["path"].values
ys    = labeled["y"].values

# paths, ys ya definidos (por archivo, no por segmento)
paths_train, paths_tmp, y_train, y_tmp = train_test_split(
    paths, ys, test_size=(TEST_SIZE + VAL_SIZE), random_state=RANDOM_SEED, stratify=ys
)

# aquí tmp es (val+test); lo partimos a la mitad si VAL_SIZE==TEST_SIZE
rel_test = TEST_SIZE / (TEST_SIZE + VAL_SIZE)

paths_val, paths_test, y_val, y_test = train_test_split(
    paths_tmp, y_tmp, test_size=rel_test, random_state=RANDOM_SEED, stratify=y_tmp
)

print("Train/Val/Test:", len(paths_train), "/", len(paths_val), "/", len(paths_test))

Train/Val/Test: 892 / 191 / 192


## Celda 5 — Extraer MFCC por segmento y “expandir” etiquetas a segmentos

In [7]:
def process_labeled_file_list(file_list, label_list):
    feats = []
    labs = []
    file_index = []  # trazabilidad: (archivo, segmento)
    for p, y in zip(file_list, label_list):
        x = load_and_clean(p)
        segs = segment_signal(x, SR, SEG_SEC, HOP_SEC)
        for k, seg in enumerate(segs):
            feats.append(mfcc_features(seg))
            labs.append(y)
            file_index.append({"file": str(p), "segment": int(k)})
    X = np.stack(feats, axis=0)  # (Nseg, C, n_mfcc, T)
    y = np.array(labs, dtype=np.int64)
    return X, y, file_index

print("Extrayendo TRAIN...")
X_train, y_train_seg, idx_train = process_labeled_file_list(paths_train, y_train)
print("Extrayendo VAL...")
X_val, y_val_seg, idx_val = process_labeled_file_list(paths_val, y_val)
print("Extrayendo TEST...")
X_test, y_test_seg, idx_test = process_labeled_file_list(paths_test, y_test)

print("Shapes:")
print("X_train:", X_train.shape, "y:", y_train_seg.shape)
print("X_val:", X_val.shape, "y:", y_val_seg.shape)
print("X_test:", X_test.shape, "y:", y_test_seg.shape)

Extrayendo TRAIN...


c:\Users\leona\.conda\envs\bees_rt\lib\site-packages\librosa\core\intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Extrayendo VAL...
Extrayendo TEST...
Shapes:
X_train: (52551, 3, 32, 201) y: (52551,)
X_val: (11253, 3, 32, 201) y: (11253,)
X_test: (11308, 3, 32, 201) y: (11308,)


### CELDA 5.5 - VALIDACIÓN DEL PREPROCESAMIENTO

In [8]:
print("="*60)
print("VALIDACIÓN DEL PREPROCESAMIENTO")
print("="*60)

# Tomar 2 archivos de ejemplo para validar
sample_paths = paths_train[:2]
sample_labels = y_train[:2]

for i, (p, label) in enumerate(zip(sample_paths, sample_labels)):
    print(f"\n--- Ejemplo {i+1}: {Path(p).name} (clase={label}) ---")
    
    # 1. Load sin trim
    x_raw, _ = librosa.load(str(p), sr=SR, mono=True)
    print(f"  Audio original: {len(x_raw)} samples ({len(x_raw)/SR:.2f}s), peak={np.max(np.abs(x_raw)):.4f}")
    
    # 2. Trim
    x_trimmed, _ = librosa.effects.trim(x_raw, top_db=TRIM_DB)
    print(f"  Después de trim: {len(x_trimmed)} samples ({len(x_trimmed)/SR:.2f}s), peak={np.max(np.abs(x_trimmed)):.4f}")
    
    # 3. Peak normalization
    x_norm = peak_normalize(x_trimmed)
    print(f"  Después de peak norm: peak={np.max(np.abs(x_norm)):.4f}")
    
    # 4. Segmentar
    segs = segment_signal(x_norm, SR, SEG_SEC, HOP_SEC)
    print(f"  Segmentos generados: {len(segs)}")
    
    # 5. MFCC del primer segmento
    if len(segs) > 0:
        # Extraer con 33 coefs
        mfcc_raw = librosa.feature.mfcc(
            y=segs[0], sr=SR, n_mfcc=33, n_fft=N_FFT, hop_length=HOP_LEN,
            fmin=FMIN, fmax=FMAX
        )
        print(f"  MFCC raw (con C0): {mfcc_raw.shape}")
        
        # Eliminar C0
        mfcc_no_c0 = mfcc_raw[1:, :]
        print(f"  MFCC sin C0: {mfcc_no_c0.shape}")
        
        # Deltas
        d1 = librosa.feature.delta(mfcc_no_c0)
        d2 = librosa.feature.delta(mfcc_no_c0, order=2)
        feat_final = np.stack([mfcc_no_c0, d1, d2], axis=0)
        print(f"  Feature final (MFCC + Δ + ΔΔ): {feat_final.shape}")
        
        # Validar shape
        assert feat_final.shape[0] == 3, "Canal incorrecto"
        assert feat_final.shape[1] == 32, "n_mfcc incorrecto (debe ser 32 sin C0)"
        print("  ✓ Shape validado: (3, 32, T)")

print("\n" + "="*60)
print("VALIDACIÓN COMPLETADA")
print("="*60)

VALIDACIÓN DEL PREPROCESAMIENTO

--- Ejemplo 1: 2022-07-14--21-19-04_1__segment5.wav (clase=3) ---
  Audio original: 960000 samples (60.00s), peak=0.1137
  Después de trim: 960000 samples (60.00s), peak=0.1137
  Después de peak norm: peak=1.0000
  Segmentos generados: 59
  MFCC raw (con C0): (33, 201)
  MFCC sin C0: (32, 201)
  Feature final (MFCC + Δ + ΔΔ): (3, 32, 201)
  ✓ Shape validado: (3, 32, T)

--- Ejemplo 2: 2022-07-06--18-10-58_1__segment0.wav (clase=3) ---
  Audio original: 960000 samples (60.00s), peak=1.0147
  Después de trim: 960000 samples (60.00s), peak=1.0147
  Después de peak norm: peak=1.0000
  Segmentos generados: 59
  MFCC raw (con C0): (33, 201)
  MFCC sin C0: (32, 201)
  Feature final (MFCC + Δ + ΔΔ): (3, 32, 201)
  ✓ Shape validado: (3, 32, T)

VALIDACIÓN COMPLETADA


## Celda 6 — Guardar ```.npy``` + metadatos

In [9]:
# Para reducir memoria en disco:
X_train = X_train.astype(np.float16)
X_val   = X_val.astype(np.float16)
X_test  = X_test.astype(np.float16)

np.save(OUT_DIR / "X_train.npy", X_train)
np.save(OUT_DIR / "y_train.npy", y_train_seg)
np.save(OUT_DIR / "X_val.npy",   X_val)
np.save(OUT_DIR / "y_val.npy",   y_val_seg)
np.save(OUT_DIR / "X_test.npy",  X_test)
np.save(OUT_DIR / "y_test.npy",  y_test_seg)

with open(OUT_DIR / "files_train.json", "w") as f:
    json.dump(idx_train, f, indent=2)
with open(OUT_DIR / "files_val.json", "w") as f:
    json.dump(idx_val, f, indent=2)
with open(OUT_DIR / "files_test.json", "w") as f:
    json.dump(idx_test, f, indent=2)

class_meaning = {
    0: "original / con reina funcional",
    1: "no presente",
    2: "presente y rechazada",
    3: "presente y recién aceptada"
}

with open(OUT_DIR / "meta.json", "w") as f:
    json.dump({
        "data_dir": str(DATA_DIR),
        "csv_path": str(CSV_PATH),
        "id_col": ID_COL,
        "target_col": TARGET_COL,
        "classes": [0, 1, 2, 3],
        "class_meaning": class_meaning,
        
        # ===== CONTRATO DE PREPROCESAMIENTO =====
        "preprocessing_pipeline": [
            "1. load + resample to sr=16000",
            "2. trim(top_db=30)",
            "3. peak_normalization (per file, after trim)",
            "4. segment(duration=2.0s, hop=1.0s)",
            "5. mfcc extraction per segment"
        ],
        "sr": SR,
        "trim_db": TRIM_DB,
        "peak_normalization": "per_file_after_trim",
        "segment_sec": SEG_SEC,
        "segment_hop_sec": HOP_SEC,
        
        # ===== MFCC CONFIGURATION =====
        "n_fft": N_FFT,
        "hop_length": HOP_LEN,
        "fmin": FMIN,
        "fmax": FMAX,
        "n_mfcc_raw": 33,
        "c0_removed": True,
        "n_mfcc_final": 32,
        "deltas": ADD_DELTAS,
        "delta_delta": ADD_DELTAS,
        
        # ===== FINAL SHAPE PER SEGMENT =====
        "feature_shape": [3, 32, 201],
        "feature_shape_description": "(channels, n_mfcc, time_frames) where channels=[mfcc, delta, delta2]",
        
        "random_seed": RANDOM_SEED
    }, f, indent=2, ensure_ascii=False)